# Prompt Versioning

Version prompts in the Opik Prompt Library, compare versions for hallucination with side-by-side experiments, and run traced inference against the winning prompt version via LiteLLM.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-ml/opik-examples/blob/main/guides/prompt_versioning/prompt_versioning.ipynb)

**The mental model:** one prompt **name** maps to many immutable **commits**. Nothing is ever overwritten — every `create_prompt()` call against the same name adds a new commit and keeps every prior one around, fetchable by hash. That gives you a simple loop: commit a candidate version, **compare** it against the current one, **promote** whichever wins, then **run** the latest commit in your application.

We follow a single prompt — an earnings-call summarizer — through that entire loop below.

**What you'll learn:**

- How to commit prompt versions with descriptive `change_description` labels using `client.create_prompt()`
- How history is retained — fetching an old commit by hash still returns it unchanged, even after newer commits exist
- How to run side-by-side prompt evaluation (`evaluate_prompt`) with an LLM-as-judge `Hallucination` metric to compare prompt versions as experiments in the Opik UI
- How to run inference against the latest commit — no hardcoded prompt text — traced with `@opik.track(project_name=...)`


In [ ]:
%pip install --quiet --upgrade opik litellm

In [ ]:
import os
import litellm
import opik
from opik.evaluation import evaluate_prompt
from opik.evaluation.metrics import Hallucination

OPIK_PROJECT_NAME = "prompt-versioning"

# Credentials come from OPIK_API_KEY / OPIK_WORKSPACE in the environment; targets Opik Cloud.
# install_mcp=False keeps configure non-interactive for headless environments.
opik.configure(project_name=OPIK_PROJECT_NAME, install_mcp=False)

client = opik.Opik()

# CI sets OPIK_EXAMPLES_MODEL to a cheap model.
MODEL_NAME = os.environ.get("OPIK_EXAMPLES_MODEL", "openai/gpt-5.6-sol")

PROMPT_NAME = "earnings-call-summarizer"


## 1. Committing prompt versions

Every call to `client.create_prompt()` with the same `name` creates an immutable version ("commit") of that prompt in the Opik Prompt Library — one name, many commits, nothing overwritten.

We pass `change_description=...` to document why each version was created — this renders directly in the Opik UI.

Note the name: `earnings-call-summarizer`, not `earnings-call-summarizer-v1`. The version lives in the commit, not the name — baking a version number into the name works against the whole model.


In [ ]:
SUMMARIZER_V1 = """You are a financial analyst summarizing earnings calls.
Provide a comprehensive summary including key metrics, guidance, and management commentary."""

# Commit v1
v1_prompt = client.create_prompt(
    name=PROMPT_NAME,
    prompt=SUMMARIZER_V1,
    change_description="Loose baseline: generic summarization instructions",
)
print(f"Created '{PROMPT_NAME}' commit v1: {v1_prompt.commit}")


In [ ]:
SUMMARIZER_V2 = """You are a financial analyst creating earnings call summaries for compliance-reviewed reports.

## Strict Rules
- ONLY include facts explicitly stated in the provided transcript
- Use EXACT numbers - never round or approximate
- Never infer sentiment not directly expressed by management
- If guidance wasn't mentioned, state "No guidance provided"
- Attribute all quotes: "CEO [Name] stated..."

## Output Format
**Reported Metrics**: [Only numbers explicitly stated]
**Management Commentary**: [Direct quotes or close paraphrases only]
**Forward Guidance**: [Only if explicitly provided]
**NOT MENTIONED**: [List key items not covered]

If uncertain whether something was stated, DO NOT include it."""

# Commit v2 — same name, new commit. v1 is not touched or replaced.
v2_prompt = client.create_prompt(
    name=PROMPT_NAME,
    prompt=SUMMARIZER_V2,
    change_description="Strict, facts-only rewrite with compliance formatting rules",
)
print(f"Created '{PROMPT_NAME}' commit v2: {v2_prompt.commit}")


Two commits now exist under the same name. Nothing was overwritten — `v1_prompt.commit` still resolves to the exact text we committed first.


## 2. Proving history is retained

Committing v2 didn't touch v1. Fetching v1 by its original commit hash — even after newer commits exist — still returns it exactly as it was.


In [ ]:
historical_v1 = client.get_prompt(name=PROMPT_NAME, commit=v1_prompt.commit)
print(f"Fetched historical v1 commit: {historical_v1.commit}")
print(f"Historical v1 description: {historical_v1.change_description}")
assert historical_v1.prompt == SUMMARIZER_V1
print("v1 is unchanged and still retrievable by hash.")


## 3. Comparing versions as experiments

Before promoting a new prompt version, evaluate both versions on an Opik dataset using `evaluate_prompt()` and an LLM-as-judge metric like `Hallucination` — summary vs. source transcript is exactly what this metric is built to catch.

Each call to `evaluate_prompt()` logs a named **experiment** in Opik linked to that specific prompt commit. Open both experiments in the Opik UI to compare hallucination scores side-by-side — v2 should score lower.


In [ ]:
DATASET_NAME = "earnings-call-summarizer-eval"

TRANSCRIPT = (
    "Apple reported Q4 revenue of $89.5 billion, up 6% year-over-year. iPhone revenue grew "
    "10% to $43.8 billion. CEO Tim Cook said 'We\'re thrilled with the strong demand for "
    "iPhone 15 Pro.'"
)
CONTEXT = "Q4 revenue: $89.5B, +6% YoY. iPhone: $43.8B, +10%. Tim Cook commented on iPhone 15 Pro demand."
QUERY = f"Summarize this earnings call:\n{TRANSCRIPT}"

dataset = client.get_or_create_dataset(name=DATASET_NAME, project_name=OPIK_PROJECT_NAME)
dataset.insert([{"input": QUERY, "context": CONTEXT}])


def score_prompt_version(prompt: opik.Prompt, experiment_name: str):
    return evaluate_prompt(
        dataset=dataset,
        messages=[
            {"role": "system", "content": prompt.prompt},
            {"role": "user", "content": "{{input}}"},
        ],
        model=MODEL_NAME,
        scoring_metrics=[Hallucination(model=MODEL_NAME)],
        experiment_name=experiment_name,
        prompt=prompt,
        project_name=OPIK_PROJECT_NAME,
    )


exp1 = score_prompt_version(v1_prompt, "earnings-call-summarizer-v1-baseline")
exp2 = score_prompt_version(v2_prompt, "earnings-call-summarizer-v2-strict")

print("Evaluations completed. Compare experiment scores side-by-side in the Opik UI.")


## 4. Running the latest commit

v2 wins on hallucination, so it's the version to ship. `client.get_prompt(name=...)` with no `commit` resolves to whatever is currently latest — here, v2 — and we pass that text straight to LiteLLM on a brand-new transcript, traced with `@opik.track(project_name=...)`.

Because the application fetches by name instead of hardcoding a string template, promoting a v3 later changes what this cell runs with zero code changes.


In [ ]:
@opik.track(project_name=OPIK_PROJECT_NAME)
def run_inference(system_prompt: str, user_query: str) -> str:
    response = litellm.completion(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
    )
    return response.choices[0].message.content


NEW_TRANSCRIPT = (
    "Microsoft reported Q2 revenue of $62.0 billion, up 18% year-over-year. Azure and other "
    "cloud services revenue grew 30%. CFO Amy Hood said guidance for next quarter assumes "
    "continued double-digit cloud growth."
)
NEW_QUERY = f"Summarize this earnings call:\n{NEW_TRANSCRIPT}"

# No commit passed -> resolves to whatever is latest right now (v2)
current_prompt = client.get_prompt(name=PROMPT_NAME)
answer = run_inference(current_prompt.prompt, NEW_QUERY)

print(f"Using prompt '{PROMPT_NAME}' commit {current_prompt.commit[:8]}...\n")
print(answer)


## Summary

| Action | SDK Method | Key Feature |
|---|---|---|
| Commit a version | `client.create_prompt(name=..., prompt=..., change_description=...)` | New immutable commit hash; `change_description` labels its purpose in the Opik UI |
| Fetch a specific commit | `client.get_prompt(name=..., commit="<hash>")` | Retrieves exact historical prompt text, unaffected by later commits |
| Compare as experiments | `evaluate_prompt(..., prompt=prompt, experiment_name=...)` | Each call creates a distinct experiment linked to that commit, for side-by-side comparison |
| Run the latest commit | `client.get_prompt(name=...)` (no `commit`) + `@opik.track(...)` | Resolves to whatever is newest — application code never hardcodes prompt text |

**Key takeaways:**

1. Prompts are immutable — calling `create_prompt` with an existing name creates a new commit hash, and every prior commit stays retrievable by hash. Don't bake the version into the name (e.g. avoid `earnings-call-summarizer-v1`); one name, many commits.
2. Compare before you promote: `evaluate_prompt()` ties each experiment to a specific commit, so you decide which version wins with data, not guesswork.
3. Decouple application code from prompt text by fetching the latest commit dynamically via `client.get_prompt()`, so the version you evaluated is the version you run.

**Learn more:** [Prompt management](https://www.comet.com/docs/opik/v1/prompt_engineering/prompt_management) · [Evaluation concepts (Experiments)](https://www.comet.com/docs/opik/evaluation/concepts)
